In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score,roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [8]:
data=pd.read_csv("/content/digital_marketing_campaign_dataset.csv")
df=pd.DataFrame(data)
df.head()

,CustomerID,Age,Gender,Income,CampaignChannel,CampaignType,AdSpend,ClickThroughRate,ConversionRate,WebsiteVisits,PagesPerVisit,TimeOnSite,SocialShares,EmailOpens,EmailClicks,PreviousPurchases,LoyaltyPoints,AdvertisingPlatform,AdvertisingTool,Conversion
0,8000,56,Female,136912,Social Media,Awareness,6497.870068,0.043919,0.088031,0,2.399017,7.396803,19,6,9,4,688,IsConfid,ToolConfid,1
1,8001,69,Male,41760,Email,Retention,3898.668606,0.155725,0.182725,42,2.917138,5.352549,5,2,7,2,3459,IsConfid,ToolConfid,1
2,8002,46,Female,88456,PPC,Awareness,1546.429596,0.277490,0.076423,2,8.223619,13.794901,0,11,2,8,2337,IsConfid,ToolConfid,1
3,8003,32,Female,44085,PPC,Conversion,539.525936,0.137611,0.088004,47,4.540939,14.688363,89,2,2,0,2463,IsConfid,ToolConfid,1
4,8004,60,Female,83964,PPC,Conversion,1678.043573,0.252851,0.109940,0,2.046847,13.993370,6,6,6,8,4345,IsConfid,ToolConfid,1


In [9]:
encoder=LabelEncoder()
df["Gender"]=encoder.fit_transform(df["Gender"])
df=pd.get_dummies(df,columns=["CampaignChannel","CampaignType"])
df["email_engagement"] = df["EmailClicks"] / (df["EmailOpens"] + 1)
df["engagement_score"] = (df["PagesPerVisit"]*df["TimeOnSite"])
df["customer_value"] = (df["PreviousPurchases"]*df["LoyaltyPoints"])
df["marketing_efficiency"] = (df["ClickThroughRate"]*df["TimeOnSite"])
df["spend_per_visit"] = (df["AdSpend"]/(df["WebsiteVisits"]+1))
X=df.drop(["CustomerID","Conversion","AdvertisingPlatform","AdvertisingTool"],axis=1)
Y=df["Conversion"]
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

In [12]:
model=RandomForestClassifier()
model.fit(X_train,Y_train)
y_pred = model.predict(X_test)
print(confusion_matrix(Y_test,y_pred))
print(classification_report(Y_test,y_pred))
y_prob = model.predict_proba(X_test)[:,1]

print(roc_auc_score(Y_test, y_prob))

[[  36  158]
 [   8 1398]]
              precision    recall  f1-score   support

           0       0.82      0.19      0.30       194
           1       0.90      0.99      0.94      1406

    accuracy                           0.90      1600
   macro avg       0.86      0.59      0.62      1600
weighted avg       0.89      0.90      0.87      1600

0.825123550028596


In [11]:
import pandas as pd

importance = pd.DataFrame({
    "feature": X.columns,
    "importance":model.feature_importances_
})

importance.sort_values(
    "importance",
    ascending=False
).head(20)

,feature,importance
25,customer_value,0.070709
3,AdSpend,0.067478
26,marketing_efficiency,0.065333
24,engagement_score,0.065160
5,ConversionRate,0.062472
4,ClickThroughRate,0.059431
7,PagesPerVisit,0.056704
10,EmailOpens,0.056479
13,LoyaltyPoints,0.053963
8,TimeOnSite,0.052484


In [13]:
print(df["Conversion"].value_counts(normalize=True))

Conversion
1    0.8765
0    0.1235
Name: proportion, dtype: float64


In [ ]:
# dataset heavily skewed